# L06 Matplotlib 折线与散点入门 · 标准答案

依据本仓库当前 homework.md 的三道题编写。每题的代码单元均自带导入、函数、内存验收，可在新内核中单独运行，不依赖其他题。

## 使用方式与契约
正常调用 plot_series / plot_scatter / plot_two_series 时传入 PNG 字符串路径，父目录由调用者先准备；函数会保存文件并原样返回路径。等长空序列和单点有效；长度不匹配在建图前抛 ValueError；不修改输入，不关闭调用者的图。

下方自动验收把 plt.savefig 临时替换为内存捕获器：真实渲染 PNG 到 BytesIO，同时核对调用的目标路径、图元数据、样式、标题、网格、图例和画布生命周期，不创建本地输出文件。正常保存逻辑仍在函数中保留。Notebook 中没有预填执行结果；不把待运行断言冒充已运行结果。

无界面模式在导入 pyplot 前设 Agg。Matplotlib 自身初始化可能维护字体缓存，这属于使用者环境行为；本次交付只向 GitHub 提交 Notebook，未在本地执行或写出图像。

## 方法与易错点
先做长度校验，再创建独立 Figure；try/finally 保证即便保存失败也关闭自己的图。不能用 close("all")，否则会关闭调用者图；不能先关闭再保存。题二必须使用 scatter，不能用只带圆点的 plot 冒充。题三既要 label，也要 legend 才能显示系列名。空序列仍应输出合法的空坐标图，而非不返回或报错。


## 题一：折线图函数

是什么：把等长的 x/y 按输入顺序连成实线并加圆点。为什么：保留有业务意义的顺序，便于看趋势。怎么写：长度检查→figure→plot→固定标注→保存→finally 关闭。易错：漏掉 marker；只传 y 导致业务横轴丢失；保存失败时遗留画布。何时用：题定有序序列或时间走势。


In [ ]:
from io import BytesIO
from unittest.mock import patch
from copy import deepcopy
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.colors import to_rgba
from matplotlib.collections import PathCollection
import numpy as np

def plot_series(x, y, out_path: str) -> str:
    """实线圆点折线；固定标签、标题、网格；保存关闭后返回路径。"""
    if len(x) != len(y):
        raise ValueError("x and y must have the same length")
    fig = plt.figure()
    try:
        plt.plot(x, y, color="blue", linestyle="-", marker="o")
        plt.xlabel("x")
        plt.ylabel("y")
        plt.title("series")
        plt.grid(True)
        plt.savefig(out_path, format="png")
    finally:
        plt.close(fig)
    return out_path

def check_case(fn, args, title, kind):
    """拦截输出到内存，仍使用真实 PNG 编码器渲染。"""
    original = deepcopy(args)
    before = set(plt.get_fignums())
    target = "l06_memory_check.png"
    captured = []

    def capture(path, *save_args, **save_kwargs):
        assert path == target
        assert not save_args
        assert save_kwargs == {"format": "png"}
        fig = plt.gcf()
        assert len(fig.axes) == 1
        ax = fig.axes[0]
        assert ax.get_xlabel() == "x"
        assert ax.get_ylabel() == "y"
        assert ax.get_title() == title
        assert any(line.get_visible() for line in ax.get_xgridlines())
        assert any(line.get_visible() for line in ax.get_ygridlines())

        if kind == "scatter":
            assert len(ax.lines) == 0
            assert len(ax.collections) == 1
            assert isinstance(ax.collections[0], PathCollection)
            expected = np.column_stack(args).reshape(-1, 2)
            np.testing.assert_allclose(
                ax.collections[0].get_offsets(), expected
            )
        else:
            assert len(ax.collections) == 0
            expected_count = 2 if kind == "two" else 1
            assert len(ax.lines) == expected_count
            for i, line in enumerate(ax.lines):
                np.testing.assert_allclose(line.get_xdata(), args[0])
                np.testing.assert_allclose(line.get_ydata(), args[i + 1])
                assert line.get_linestyle() == ("--" if i == 1 else "-")
                assert line.get_marker() == ("s" if i == 1 else "o")
            if kind == "two":
                for line, color, label in zip(
                    ax.lines, ["blue", "red"], ["A", "B"]
                ):
                    assert to_rgba(line.get_color()) == to_rgba(color)
                    assert line.get_label() == label
                legend = ax.get_legend()
                assert legend is not None
                assert [t.get_text() for t in legend.get_texts()] == ["A", "B"]

        stream = BytesIO()
        fig.savefig(stream, format="png")
        blob = stream.getvalue()
        assert blob.startswith(b"\x89PNG\r\n\x1a\n")
        stream.seek(0)
        image = mpimg.imread(stream, format="png")
        assert image.shape[0] > 0 and image.shape[1] > 0
        captured.append(blob)

    with patch.object(plt, "savefig", side_effect=capture) as save:
        assert fn(*args, target) == target
        assert save.call_count == 1
    assert len(captured) == 1
    assert set(plt.get_fignums()) == before
    for after, prior in zip(args, original):
        np.testing.assert_array_equal(after, prior)


def check_bad_lengths(fn, args):
    before = set(plt.get_fignums())
    with patch.object(plt, "figure") as make, patch.object(plt, "savefig") as save:
        try:
            fn(*args, "must_not_be_created.png")
        except ValueError:
            pass
        else:
            raise AssertionError("expected ValueError")
        make.assert_not_called()
        save.assert_not_called()
    assert set(plt.get_fignums()) == before


def check_save_failure(fn, args):
    before = set(plt.get_fignums())
    with patch.object(plt, "savefig", side_effect=OSError("simulated")):
        try:
            fn(*args, "not_written.png")
        except OSError:
            pass
        else:
            raise AssertionError("save error must propagate")
    assert set(plt.get_fignums()) == before

# 保留调用者图，检查本题不会误关闭其他图。
caller_fig = plt.figure()
try:
    check_case(plot_series, ([1, 2, 3], [10, 20, 15]), "series", "line")
    check_case(plot_series, ([5], [8]), "series", "line")
    check_case(plot_series, ([], []), "series", "line")
    check_case(plot_series, (np.array([3, 1, 2]), np.array([7, 4, 9])), "series", "line")
    check_bad_lengths(plot_series, ([1, 2], [10]))
    check_bad_lengths(plot_series, ([], [10]))
    check_save_failure(plot_series, ([1], [2]))
    assert plt.fignum_exists(caller_fig.number)
finally:
    plt.close(caller_fig)
print("L06 Q1: all in-memory checks passed")


## 题二：散点图函数

是什么：用 PathCollection 表达 x/y 点对，不连接观测。为什么：行号没有连接意义时，连线会制造不存在的路径。怎么写：显式调用 plt.scatter，其他输出契约与题一一致。易错：用 plot 的点标记冒充 scatter；两列分别清洗导致观测错配；由正向关系断言因果。何时用：两个变量的样本关系、离群点和重叠分布。下方单元独立于题一。


In [ ]:
from io import BytesIO
from unittest.mock import patch
from copy import deepcopy
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.colors import to_rgba
from matplotlib.collections import PathCollection
import numpy as np

def plot_scatter(x, y, out_path: str) -> str:
    """真正的散点图；固定标签、标题、网格；保存关闭后返回路径。"""
    if len(x) != len(y):
        raise ValueError("x and y must have the same length")
    fig = plt.figure()
    try:
        plt.scatter(x, y)
        plt.xlabel("x")
        plt.ylabel("y")
        plt.title("scatter")
        plt.grid(True)
        plt.savefig(out_path, format="png")
    finally:
        plt.close(fig)
    return out_path

def check_case(fn, args, title, kind):
    """拦截输出到内存，仍使用真实 PNG 编码器渲染。"""
    original = deepcopy(args)
    before = set(plt.get_fignums())
    target = "l06_memory_check.png"
    captured = []

    def capture(path, *save_args, **save_kwargs):
        assert path == target
        assert not save_args
        assert save_kwargs == {"format": "png"}
        fig = plt.gcf()
        assert len(fig.axes) == 1
        ax = fig.axes[0]
        assert ax.get_xlabel() == "x"
        assert ax.get_ylabel() == "y"
        assert ax.get_title() == title
        assert any(line.get_visible() for line in ax.get_xgridlines())
        assert any(line.get_visible() for line in ax.get_ygridlines())

        if kind == "scatter":
            assert len(ax.lines) == 0
            assert len(ax.collections) == 1
            assert isinstance(ax.collections[0], PathCollection)
            expected = np.column_stack(args).reshape(-1, 2)
            np.testing.assert_allclose(
                ax.collections[0].get_offsets(), expected
            )
        else:
            assert len(ax.collections) == 0
            expected_count = 2 if kind == "two" else 1
            assert len(ax.lines) == expected_count
            for i, line in enumerate(ax.lines):
                np.testing.assert_allclose(line.get_xdata(), args[0])
                np.testing.assert_allclose(line.get_ydata(), args[i + 1])
                assert line.get_linestyle() == ("--" if i == 1 else "-")
                assert line.get_marker() == ("s" if i == 1 else "o")
            if kind == "two":
                for line, color, label in zip(
                    ax.lines, ["blue", "red"], ["A", "B"]
                ):
                    assert to_rgba(line.get_color()) == to_rgba(color)
                    assert line.get_label() == label
                legend = ax.get_legend()
                assert legend is not None
                assert [t.get_text() for t in legend.get_texts()] == ["A", "B"]

        stream = BytesIO()
        fig.savefig(stream, format="png")
        blob = stream.getvalue()
        assert blob.startswith(b"\x89PNG\r\n\x1a\n")
        stream.seek(0)
        image = mpimg.imread(stream, format="png")
        assert image.shape[0] > 0 and image.shape[1] > 0
        captured.append(blob)

    with patch.object(plt, "savefig", side_effect=capture) as save:
        assert fn(*args, target) == target
        assert save.call_count == 1
    assert len(captured) == 1
    assert set(plt.get_fignums()) == before
    for after, prior in zip(args, original):
        np.testing.assert_array_equal(after, prior)


def check_bad_lengths(fn, args):
    before = set(plt.get_fignums())
    with patch.object(plt, "figure") as make, patch.object(plt, "savefig") as save:
        try:
            fn(*args, "must_not_be_created.png")
        except ValueError:
            pass
        else:
            raise AssertionError("expected ValueError")
        make.assert_not_called()
        save.assert_not_called()
    assert set(plt.get_fignums()) == before


def check_save_failure(fn, args):
    before = set(plt.get_fignums())
    with patch.object(plt, "savefig", side_effect=OSError("simulated")):
        try:
            fn(*args, "not_written.png")
        except OSError:
            pass
        else:
            raise AssertionError("save error must propagate")
    assert set(plt.get_fignums()) == before

caller_fig = plt.figure()
try:
    check_case(plot_scatter, ([1., 2., 3., 4.], [2.1, 3.9, 6.2, 7.8]), "scatter", "scatter")
    check_case(plot_scatter, ([5], [8]), "scatter", "scatter")
    check_case(plot_scatter, ([], []), "scatter", "scatter")
    check_case(plot_scatter, (np.array([3, 1, 2]), np.array([7, 4, 9])), "scatter", "scatter")
    check_bad_lengths(plot_scatter, ([1, 2, 3], [1, 2]))
    check_bad_lengths(plot_scatter, ([], [10]))
    check_save_failure(plot_scatter, ([1], [2]))
    assert plt.fignum_exists(caller_fig.number)
finally:
    plt.close(caller_fig)
print("L06 Q2: all in-memory checks passed")


## 题三：双折线对比

是什么：同一坐标系中比较 A/B 两条折线。为什么：共同横轴和单位使走势可直接比较，线型和标记也能在黑白图中区分。怎么写：三者长度同时比较，连续两次 plot，设置 label，再 legend。易错：只校验 x/y1 忽略 y2；只有 label 没有 legend；创建两张图导致无法同图比较。何时用：同单位、同横轴的两个系列。下方单元独立于前两题。


In [ ]:
from io import BytesIO
from unittest.mock import patch
from copy import deepcopy
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.colors import to_rgba
from matplotlib.collections import PathCollection
import numpy as np

def plot_two_series(x, y1, y2, out_path: str) -> str:
    """A 为蓝实线圆点，B 为红虚线方点；显示图例。"""
    if not (len(x) == len(y1) == len(y2)):
        raise ValueError("x, y1 and y2 must have the same length")
    fig = plt.figure()
    try:
        plt.plot(x, y1, color="blue", linestyle="-", marker="o", label="A")
        plt.plot(x, y2, color="red", linestyle="--", marker="s", label="B")
        plt.xlabel("x")
        plt.ylabel("y")
        plt.title("A vs B")
        plt.grid(True)
        plt.legend()
        plt.savefig(out_path, format="png")
    finally:
        plt.close(fig)
    return out_path

def check_case(fn, args, title, kind):
    """拦截输出到内存，仍使用真实 PNG 编码器渲染。"""
    original = deepcopy(args)
    before = set(plt.get_fignums())
    target = "l06_memory_check.png"
    captured = []

    def capture(path, *save_args, **save_kwargs):
        assert path == target
        assert not save_args
        assert save_kwargs == {"format": "png"}
        fig = plt.gcf()
        assert len(fig.axes) == 1
        ax = fig.axes[0]
        assert ax.get_xlabel() == "x"
        assert ax.get_ylabel() == "y"
        assert ax.get_title() == title
        assert any(line.get_visible() for line in ax.get_xgridlines())
        assert any(line.get_visible() for line in ax.get_ygridlines())

        if kind == "scatter":
            assert len(ax.lines) == 0
            assert len(ax.collections) == 1
            assert isinstance(ax.collections[0], PathCollection)
            expected = np.column_stack(args).reshape(-1, 2)
            np.testing.assert_allclose(
                ax.collections[0].get_offsets(), expected
            )
        else:
            assert len(ax.collections) == 0
            expected_count = 2 if kind == "two" else 1
            assert len(ax.lines) == expected_count
            for i, line in enumerate(ax.lines):
                np.testing.assert_allclose(line.get_xdata(), args[0])
                np.testing.assert_allclose(line.get_ydata(), args[i + 1])
                assert line.get_linestyle() == ("--" if i == 1 else "-")
                assert line.get_marker() == ("s" if i == 1 else "o")
            if kind == "two":
                for line, color, label in zip(
                    ax.lines, ["blue", "red"], ["A", "B"]
                ):
                    assert to_rgba(line.get_color()) == to_rgba(color)
                    assert line.get_label() == label
                legend = ax.get_legend()
                assert legend is not None
                assert [t.get_text() for t in legend.get_texts()] == ["A", "B"]

        stream = BytesIO()
        fig.savefig(stream, format="png")
        blob = stream.getvalue()
        assert blob.startswith(b"\x89PNG\r\n\x1a\n")
        stream.seek(0)
        image = mpimg.imread(stream, format="png")
        assert image.shape[0] > 0 and image.shape[1] > 0
        captured.append(blob)

    with patch.object(plt, "savefig", side_effect=capture) as save:
        assert fn(*args, target) == target
        assert save.call_count == 1
    assert len(captured) == 1
    assert set(plt.get_fignums()) == before
    for after, prior in zip(args, original):
        np.testing.assert_array_equal(after, prior)


def check_bad_lengths(fn, args):
    before = set(plt.get_fignums())
    with patch.object(plt, "figure") as make, patch.object(plt, "savefig") as save:
        try:
            fn(*args, "must_not_be_created.png")
        except ValueError:
            pass
        else:
            raise AssertionError("expected ValueError")
        make.assert_not_called()
        save.assert_not_called()
    assert set(plt.get_fignums()) == before


def check_save_failure(fn, args):
    before = set(plt.get_fignums())
    with patch.object(plt, "savefig", side_effect=OSError("simulated")):
        try:
            fn(*args, "not_written.png")
        except OSError:
            pass
        else:
            raise AssertionError("save error must propagate")
    assert set(plt.get_fignums()) == before

caller_fig = plt.figure()
try:
    check_case(plot_two_series, ([1, 2, 3, 4], [10, 11, 13, 12], [9, 10, 10, 12]), "A vs B", "two")
    check_case(plot_two_series, ([1], [2], [3]), "A vs B", "two")
    check_case(plot_two_series, ([], [], []), "A vs B", "two")
    check_case(plot_two_series, (np.array([2, 1]), np.array([8, 7]), np.array([6, 5])), "A vs B", "two")
    check_bad_lengths(plot_two_series, ([1, 2], [10], [20, 30]))
    check_bad_lengths(plot_two_series, ([1, 2], [10, 20], [30]))
    check_bad_lengths(plot_two_series, ([1], [10, 20], [30, 40]))
    check_save_failure(plot_two_series, ([1], [2], [3]))
    assert plt.fignum_exists(caller_fig.number)
finally:
    plt.close(caller_fig)
print("L06 Q3: all in-memory checks passed")


## 复杂度与验收说明

每个函数对 n 个点构造图元，数据处理部分时间/空间均为 O(n)，实际渲染还受输出像素、后端与文字布局影响。PNG 文件“存在且非空”只是最低要求；上述内存测试进一步核对实际 PNG 解码和图元语义。需落盘核验时，由使用者在已存在的输出目录调用函数，再检查真实文件；本次交付未运行落盘测试，未预填测试通过记录。
